# Health Diagnosis: Brain Tumor MRI Detection & Classification
### Deep Learning Architecture: EfficientNet-B0 (PyTorch / timm)
**Target Classes:** `GLIOMA`, `MENINGIOMA`, `HEALTHY`, `PITUITARY`
---
This notebook contains the complete PyTorch pipeline for Brain Cancer detection from MRI scans.

In [ ]:
import os
import torch
import timm
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms

# Configure Device (GPU / CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {DEVICE}")

## 1. Load Trained EfficientNet-B0 Model
Loads weights from `weights/efficientnet_b0.pth` trained with weighted cross entropy.

In [ ]:
CLASSES = ["GLIOMA", "MENINGIOMA", "HEALTHY", "PITUITARY"]
weights_candidates = [
    os.path.join("..", "weights", "efficientnet_b0.pth"),
    os.path.join("weights", "efficientnet_b0.pth")
]
weights_path = next((p for p in weights_candidates if os.path.exists(p)), None)

model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=4).to(DEVICE)
if weights_path:
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    print(f"Loaded weights from {weights_path}")
else:
    print("Pre-trained checkpoint not found at paths, initialized standard head.")
model.eval()

## 2. MRI Preprocessing & Inference Function

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_brain_mri(image_path):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probabilities = torch.nn.functional.softmax(logits, dim=1)[0]
        confidence, pred_idx = torch.max(probabilities, 0)
    pred_class = CLASSES[pred_idx.item()]
    conf_val = float(confidence.item()) * 100
    all_scores = {cls: round(float(probabilities[i]) * 100, 2) for i, cls in enumerate(CLASSES)}
    return pred_class, conf_val, all_scores, image

## 3. Visual Demonstration on Sample MRI Scans

In [ ]:
sample_dirs = [os.path.join("frontend", "public", "samples"), os.path.join("..", "frontend", "public", "samples")]
base_dir = next((d for d in sample_dirs if os.path.exists(d)), ".")
test_samples = ["glioma.jpg", "meningioma.jpg", "pituitary.jpg", "healthy.jpg"]

fig, axes = plt.subplots(1, len(test_samples), figsize=(18, 5))
for idx, sample_name in enumerate(test_samples):
    sample_path = os.path.join(base_dir, sample_name)
    if os.path.exists(sample_path):
        pred, conf, breakdown, img = predict_brain_mri(sample_path)
        axes[idx].imshow(img)
        axes[idx].set_title(f"{pred} ({conf:.1f}%)", fontsize=12, fontweight="bold")
        axes[idx].axis("off")
    else:
        axes[idx].text(0.5, 0.5, f"{sample_name} missing", ha="center")
plt.tight_layout()
plt.show()